![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 22 -- Lab 2: Clothes with a CNN

An Olympiad-style image task: 70,000 photos of clothes from an online shop, 28 x 28 pixels, ten classes (T-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot), scored by accuracy. You first run the Week 3 recipe (flatten, then a 784-128-10 network) as the baseline, then build the deck's CNN from `Conv2d`, `ReLU`, `MaxPool2d`, `Flatten` and `Linear`, trace its shapes before training, train it with the Day 16 loop, look at its kernels and feature maps, test the best checkpoint, and write a submission file.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Fashion-MNIST (Xiao, Rasul and Vollgraf, 2017), downloaded by `torchvision.datasets.FashionMNIST` in the setup cell (about 30 MB): 60,000 training and 10,000 test images with labels 0 to 9. You split the training part into 50,000 for training and 10,000 for validation. Turn on a GPU in Colab (Runtime, Change runtime type, T4) if you can; on a CPU each epoch of the CNN takes about a minute.

---
# Setup

In [ ]:
# torch, torchvision, matplotlib, pandas and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q torch torchvision matplotlib pandas scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])   # mean and spread of the training pixels
full_train = datasets.FashionMNIST(root="data", train=True, download=True, transform=tf)
test_set = datasets.FashionMNIST(root="data", train=False, download=True, transform=tf)
CLASSES = full_train.classes
img, label = full_train[0]
print(len(full_train), len(test_set), img.shape, img.dtype, CLASSES[label])

---
# Part 1 -- Look, split, baseline

The *An Image Is a Grid of Numbers* and *How We Did It Before* slides.

## Task 1: Look at the clothes

Show the first 10 training images in a row: `full_train[i]` gives `(img, label)`, `img[0]` is the 28 x 28 pixel matrix (`imshow(..., cmap="gray_r")`), the class name `CLASSES[label]` as the title (font size 8), axes off. Then print `img.min().item()` and `img.max().item()`: after `Normalize` the values are no longer 0 to 1.

In [ ]:
# Your code here


## Task 2: Split and load

Split `full_train` into 50,000 training and 10,000 validation images with `random_split(full_train, [50000, 10000], generator=torch.Generator().manual_seed(0))` as `train_set, val_set`. Build `train_loader` (batch 64, `shuffle=True`), `val_loader` and `test_loader` (batch 1000, no shuffle). Take one batch `xb, yb = next(iter(train_loader))` and print both shapes: `(64, 1, 28, 28)` and `(64,)`: four axes, channels first, nothing flattened.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_set) == 50000 and xb.shape == (64, 1, 28, 28) and yb.shape == (64,), 'split: 50,000 / 10,000; batches of 64 images of shape (1, 28, 28)'
print('Task 2 passed')

The two functions below are Day 16's: `evaluate` scores a model on a loader, `fit` runs the four moves with a validation pass every epoch and keeps the best epoch's weights in a file. They work for any model that maps a batch to 10 scores.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total += loss_fn(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n

def fit(model, epochs, lr, checkpoint):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, best_acc = [], 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)          # 1. predict  2. loss
            opt.zero_grad(); loss.backward()       # 3. gradient
            opt.step()                             # 4. update
        val_loss, val_acc = evaluate(model, val_loader)
        history.append(val_acc)
        if val_acc > best_acc:
            best_acc = val_acc; torch.save(model.state_dict(), checkpoint)
        print(f"epoch {epoch}: validation loss {val_loss:.3f}, accuracy {val_acc:.3f}")
    return history

## Task 3: The Week 3 baseline

Build `mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)`, print its parameter count (expected 101,770), and train it with `hist_mlp = fit(mlp, 3, 1e-3, "best_mlp.pt")`. Expected validation accuracy after 3 epochs: about 0.86 to 0.88.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert sum(p.numel() for p in mlp.parameters()) == 101770 and max(hist_mlp) > 0.8, 'baseline: Flatten, Linear(784, 128), ReLU, Linear(128, 10); 3 epochs with Adam 1e-3'
print('Task 3 passed')

---
# Part 2 -- The CNN

The *Pipeline, With Shapes*, *In Code* and *Counting Its Dials* slides of the CNN section.

## Task 4: Build the CNN

Write `make_cnn()` returning `nn.Sequential` with, in order: `nn.Conv2d(1, 16, 3, padding=1)`, `nn.ReLU()`, `nn.MaxPool2d(2)`, `nn.Conv2d(16, 32, 3, padding=1)`, `nn.ReLU()`, `nn.MaxPool2d(2)`, `nn.Flatten()`, `nn.Linear(32 * 7 * 7, 10)`. Create `cnn = make_cnn().to(device)` and print it.

In [ ]:
# Your code here


## Task 5: Trace the shapes before training

The shape habit of the deck: start from `h = torch.zeros(1, 1, 28, 28).to(device)` and, in a loop over `cnn`, apply each layer and print the layer's class name with `tuple(h.shape)`. You should read `(1, 16, 28, 28)`, `(1, 16, 14, 14)`, `(1, 32, 14, 14)`, `(1, 32, 7, 7)`, `(1, 1568)`, `(1, 10)` along the way. Keep the final shape as `final_shape`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert final_shape == (1, 10), 'shape trace: apply the layers one by one; the Linear layer must receive 1568 = 32 * 7 * 7 numbers'
print('Task 5 passed')

## Task 6: Count the dials

Print the parameter count of `cnn` as `n_cnn` (expected 20,490) and, layer by layer, `sum(p.numel() for p in layer.parameters())` for each layer of `cnn`: 160, 0, 0, 4,640, 0, 0, 0, 15,690, as on the *Counting Its Dials* slide.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_cnn == 20490, 'dials: 160 + 4640 + 15690; check kernel_size=3, padding=1 and Linear(1568, 10)'
print('Task 6 passed')

## Task 7: Train it with the same loop

Run `hist_cnn = fit(cnn, 3, 1e-3, "best_cnn.pt")`: the loop of Task 3, untouched, now with convolutions inside the model. Then plot `hist_mlp` and `hist_cnn` against the epoch (markers, a legend, axis labels, a title). Expected: the CNN about 0.89 after 3 epochs, ahead of the baseline (about 0.87) with a fifth of its dials.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert max(hist_cnn) > 0.85, 'CNN training: fit(cnn, 3, 1e-3, ...) with the loaders of Task 2'
print('Task 7 passed')

---
# Part 3 -- Inside the CNN

The *What the Kernels Become* and *Hand-Made Kernels* slides: the kernels your network chose, and what they do to one image.

## Task 8: The learned kernels

Take the first layer's weights, `cnn[0].weight.detach().cpu().numpy()` (shape `(16, 1, 3, 3)`), and show the 16 kernels `W[k, 0]` as 3 x 3 images in a row (`cmap="RdBu_r"`, `vmin=-W.max(), vmax=W.max()` so red is positive and blue negative, axes off). Do some look like the edge kernels of Lab 1?

In [ ]:
# Your code here


## Task 9: The feature maps of one image

Take the first test image `x1 = test_set[0][0][None].to(device)` (shape `(1, 1, 28, 28)`) and compute the output of the first block, `fm = cnn[2](cnn[1](cnn[0](x1)))`, inside `torch.no_grad()`: shape `(1, 16, 14, 14)`. Show the image and its 16 feature maps `fm[0, k]` (`.cpu()`, `cmap="gray"`, axes off) in one row. Each map is where one kernel's pattern was found, after ReLU and pooling.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(fm.shape) == (1, 16, 14, 14), 'feature maps: Conv2d, then ReLU, then MaxPool2d, on a (1, 1, 28, 28) input'
print('Task 9 passed')

---
# Part 4 -- Test and submit

An Olympiad hand-in: the checkpoint with the best validation score, tested once, and a submission file checked before hand-in.

## Task 10: Test the best checkpoint

Create `best = make_cnn().to(device)`, load `best.load_state_dict(torch.load("best_cnn.pt", map_location=device))`, and evaluate it on `test_loader` as `test_loss, test_acc`. Print both; the accuracy should be close to the validation one.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert test_acc > 0.85, 'checkpoint: load_state_dict(torch.load("best_cnn.pt")), then evaluate on test_loader'
print('Task 10 passed')

## Task 11: Where it fails

Collect the predictions on the whole test set (`argmax` of the logits, inside `torch.no_grad()`, batch by batch, `.cpu()`) into a NumPy array `pred_test`, take `y_test = test_set.targets.numpy()`, and print `confusion_matrix(y_test, pred_test)` with the class names. Which two classes are confused most, and does that make sense for 28 x 28 pictures?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pred_test.shape == (10000,) and (pred_test == y_test).mean() > 0.85, 'predictions: one label per test image, in test order'
print('Task 11 passed')

## Task 12: Write the submission

Save `pd.DataFrame({"id": np.arange(10000), "label": pred_test})` to `submission.csv` with `index=False`, read it back, and print its shape and first 5 rows. The file must have exactly 10,000 rows and integer labels 0 to 9.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert check.shape == (10000, 2) and set(check['label'].unique()) <= set(range(10)), 'submission.csv: 10,000 rows, columns id and label'
print('Task 12 passed')

---
## Task 13: Reflect

The CNN beat the flattened network with a fifth of the dials. Which two properties of a convolutional layer explain that (the *Today's Idea* slide), and which one did the shift test of Lab 1 show? Task 5 printed `(1, 1568)` before the Linear layer. What would the error have been if `make_cnn` had said `nn.Linear(32 * 14 * 14, 10)`, and at which line would you have caught it? The confusion matrix mixes two particular classes. Name one change to the model or the data from this week's plan that could help, and one number you would watch to know whether it did.

*Your answers here*

---
## Conclusion

- **Images as tensors**: batches of shape (64, 1, 28, 28), channels first, normalised, never flattened before the convolutions.
- **The CNN**: two blocks of `Conv2d`, `ReLU`, `MaxPool2d`, then `Flatten` and `Linear`, traced shape by shape before training and counted at 20,490 dials.
- **The same loop**: Day 16's `fit` trained it unchanged, and the validation curve put it ahead of the 101,770-dial baseline.
- **Inside and out**: the learned 3 x 3 kernels and their feature maps, the test score of the best checkpoint, the confusion matrix, and a checked submission file.

---

Made By **Sattam Altwaim**